<a href="https://colab.research.google.com/github/Raksh1707/taskdeeplearning/blob/main/task15.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import torch
import torch.nn as nn
import torch.distributed as dist
import torch.multiprocessing as mp
from torch.nn.parallel import DistributedDataParallel as DDP


In [ ]:
class Model(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc = nn.Linear(10, 2)

    def forward(self, x):
        return self.fc(x)


def train(rank, world_size):
    os.environ["MASTER_ADDR"] = "127.0.0.1"
    os.environ["MASTER_PORT"] = "29500"

    # GLOO for CPU, NCCL for GPU
    backend = "nccl" if torch.cuda.is_available() else "gloo"

    dist.init_process_group(
        backend=backend,
        rank=rank,
        world_size=world_size
    )

    device = torch.device(
        f"cuda:{rank}" if torch.cuda.is_available() else "cpu"
    )

    model = Model().to(device)
    model = DDP(model)

    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    loss_fn = nn.CrossEntropyLoss()

    # Dummy data
    x = torch.randn(100, 10).to(device)
    y = torch.randint(0, 2, (100,)).to(device)

    for epoch in range(5):

        optimizer.zero_grad()

        output = model(x)
        loss = loss_fn(output, y)

        loss.backward()
        optimizer.step()

        if rank == 0:
            print(f"Epoch {epoch + 1}, Loss: {loss.item():.4f}")

    dist.destroy_process_group()




In [ ]:
world_size = 2

mp.start_processes(
    train,
    args=(world_size,),
    nprocs=world_size,
    start_method="fork",
    join=True
)

Epoch 1, Loss: 0.7864
Epoch 2, Loss: 0.7853
Epoch 3, Loss: 0.7842
Epoch 4, Loss: 0.7832
Epoch 5, Loss: 0.7821
